# Zero-Shot Cross-Lingual Transfer — Reference Solution

**Challenge**: Predict answers for 5 unseen Indic languages (Gujarati, Kannada,
Malayalam, Odia, Punjabi) using only labelled data from 6 seen languages
(Bengali, English, Hindi, Marathi, Tamil, Telugu).

**Strategy**: Cross-lingual semantic retrieval + multilingual few-shot prompting.
We build a multilingual embedding index over `train.csv`, retrieve the most
semantically similar seen-language examples for each unseen-language question,
and use them as few-shot context for a multilingual LLM.

- Reads: `./dataset/public/train.csv`, `./dataset/public/test.csv`
- Writes: `./working/submission.csv`
- Runtime: ~25 min on Kaggle T4 GPU

## 0. Setup

In [ ]:
# !pip install -q transformers accelerate sentencepiece sentence-transformers

import os, re, random
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

TRAIN_PATH  = Path('./dataset/public/train.csv')
TEST_PATH   = Path('./dataset/public/test.csv')
OUTPUT_PATH = Path('./working/submission.csv')
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

SEEN_LANGS   = {'ben', 'eng', 'hin', 'mar', 'tam', 'tel'}
UNSEEN_LANGS = {'guj', 'kan', 'mal', 'ory', 'pan'}
OPTION_COLS  = ['option_a', 'option_b', 'option_c', 'option_d']

print('Setup complete')

## 1. Load and Verify Data

In [ ]:
train_df = pd.read_csv(TRAIN_PATH, encoding='utf-8-sig')
test_df  = pd.read_csv(TEST_PATH,  encoding='utf-8-sig')

print(f'train: {len(train_df):,} rows | languages: {sorted(train_df["language"].unique())}')
print(f'test:  {len(test_df):,} rows  | languages: {sorted(test_df["language"].unique())}')

# Verify no test-language leakage in train
train_langs = set(train_df['language'].unique())
test_langs  = set(test_df['language'].unique())
assert train_langs == SEEN_LANGS,   f'Unexpected train languages: {train_langs}'
assert test_langs  == UNSEEN_LANGS, f'Unexpected test languages: {test_langs}'
assert train_langs.isdisjoint(test_langs), 'LEAK: train and test share languages!'
print('\nLanguage split verified — no overlap between train and test languages.')

In [ ]:
# Show one question from each unseen language
print('Sample unseen-language questions:')
print('=' * 60)
for lang in sorted(UNSEEN_LANGS):
    row = test_df[test_df['language'] == lang].iloc[0]
    print(f'[{lang.upper()}] {row["question"][:100]}')
    print(f'  A: {row["option_a"][:60]}')
    print()

## 2. Build Cross-Lingual Embedding Index

We embed all `train.csv` questions using a multilingual model.
At inference time, we retrieve the most semantically similar seen-language
examples for each unseen-language test question.

In [ ]:
# Multilingual embedding model — maps all scripts to a shared vector space
EMBED_MODEL = 'intfloat/multilingual-e5-small'  # ~470 MB, fast
# Alternatives (better accuracy, slower):
#   'intfloat/multilingual-e5-large'   (~2.2 GB)
#   'sentence-transformers/LaBSE'       (~1.9 GB)
#   'paraphrase-multilingual-mpnet-base-v2'

from sentence_transformers import SentenceTransformer
import torch
torch.manual_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Embedding model: {EMBED_MODEL}  |  device: {device}')
embedder = SentenceTransformer(EMBED_MODEL, device=device)

In [ ]:
# Embed training questions (prefix required by multilingual-e5)
def make_query(row):
    return f'query: {row["question"]}'

def make_passage(row):
    opts = '  '.join(f'{l}. {row[c]}' for l, c in zip('ABCD', OPTION_COLS))
    return f'passage: {row["question"]}  {opts}'

print('Embedding train questions...')
train_passages = [make_passage(r) for _, r in train_df.iterrows()]
train_embeddings = embedder.encode(
    train_passages, batch_size=64, show_progress_bar=True,
    normalize_embeddings=True, convert_to_numpy=True
)
print(f'Train embeddings shape: {train_embeddings.shape}')

In [ ]:
def retrieve_cross_lingual(query_row, k=3) -> list:
    """
    For an unseen-language question, retrieve the k most semantically similar
    examples from the seen-language training set.
    """
    q_emb = embedder.encode(
        [make_query(query_row)], normalize_embeddings=True, convert_to_numpy=True
    )
    scores = train_embeddings @ q_emb.T   # cosine similarity (both L2-normalised)
    top_k  = scores.flatten().argsort()[-k:][::-1]
    return [train_df.iloc[i].to_dict() for i in top_k]

# Test: retrieve for a Gujarati question using Hindi/Bengali examples
guj_q = test_df[test_df['language'] == 'guj'].iloc[0].to_dict()
examples = retrieve_cross_lingual(guj_q, k=2)
print(f'Query (Gujarati): {guj_q["question"][:80]}')
print(f'Retrieved ({examples[0]["language"]}): {examples[0]["question"][:80]}')

## 3. Model Configuration

In [ ]:
BACKEND   = 'local'              # 'local' | 'openai' | 'anthropic'
LLM_MODEL = 'google/gemma-2-2b-it'  # multilingual instruction model
# Alternatives: 'meta-llama/Llama-3.2-3B-Instruct', 'microsoft/Phi-3-mini-128k-instruct'

API_MODEL         = 'gpt-4o-mini'
OPENAI_API_KEY    = os.environ.get('OPENAI_API_KEY', '')
ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY', '')

FEW_SHOT_K     = 3
MAX_NEW_TOKENS = 5
BATCH_SIZE     = 8

print(f'Backend: {BACKEND}  |  LLM: {LLM_MODEL if BACKEND == "local" else API_MODEL}')

In [ ]:
model, tokenizer = None, None

if BACKEND == 'local':
    from transformers import AutoTokenizer, AutoModelForCausalLM
    torch.manual_seed(SEED)
    tokenizer = AutoTokenizer.from_pretrained(
        LLM_MODEL, trust_remote_code=True, padding_side='left'
    )
    tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL,
        torch_dtype=torch.bfloat16 if device == 'cuda' else torch.float32,
        device_map='auto' if device == 'cuda' else None,
        trust_remote_code=True,
    ).eval()
    print('LLM loaded.')

## 4. Prompt Construction and Inference

In [ ]:
def format_question(row, include_answer=False):
    lines = [f'Question: {row["question"]}']
    for letter, col in zip('ABCD', OPTION_COLS):
        lines.append(f'  {letter}. {row[col]}')
    lines.append(f'Answer: {row["answer"]}' if include_answer else 'Answer:')
    return '\n'.join(lines)

def build_prompt(test_row, few_shot_rows):
    lang_name = test_row.get('language_name', test_row['language'])
    domain    = test_row.get('domain', 'General')
    header = (
        f'You are an expert at answering {lang_name} multiple-choice questions.\n'
        f'Domain: {domain}\n'
        'The following examples are from related languages to help you understand '
        'the question format. Respond with ONLY the letter A, B, C, or D.\n\n'
    )
    examples = '\n\n'.join(format_question(r, include_answer=True) for r in few_shot_rows)
    test_q   = format_question(test_row, include_answer=False)
    return header + examples + ('\n\n' if examples else '') + test_q

In [ ]:
ANSWER_RE = re.compile(
    r'\b([ABCD])\b|\(([ABCD])\)|([ABCD])[.):]|answer\s*(?:is)?\s*:?\s*([ABCD])\b',
    re.IGNORECASE
)

def extract_answer(text):
    text = text.strip()
    if text.upper() in ('A','B','C','D'): return text.upper()
    m = ANSWER_RE.search(text)
    if m: return next(g for g in m.groups() if g).upper()
    for ch in text.upper():
        if ch in ('A','B','C','D'): return ch
    return 'A'

# Tests
for t in ['B', '(C)', 'The answer is D', 'Answer: A', 'xyz']:
    print(f'  {t!r:30s} → {extract_answer(t)}')

In [ ]:
def predict_local_batch(prompts):
    inputs = tokenizer(
        prompts, return_tensors='pt', padding=True, truncation=True, max_length=1024
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
            temperature=None, top_p=None, pad_token_id=tokenizer.pad_token_id
        )
    in_len = inputs['input_ids'].shape[1]
    return [tokenizer.decode(o[in_len:], skip_special_tokens=True) for o in outputs]

def predict_openai(prompt):
    from openai import OpenAI
    r = OpenAI(api_key=OPENAI_API_KEY).chat.completions.create(
        model=API_MODEL, messages=[{'role':'user','content':prompt}],
        max_tokens=5, temperature=0, seed=SEED
    )
    return r.choices[0].message.content.strip()

def predict_anthropic(prompt):
    import anthropic
    r = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY).messages.create(
        model=API_MODEL, max_tokens=5, messages=[{'role':'user','content':prompt}]
    )
    return r.content[0].text.strip()

print('Inference functions ready.')

## 5. Proxy Validation (Seen Language as Stand-in for Unseen)

We hold out **Telugu** from the few-shot pool and treat it as a proxy unseen
language. This measures how well our cross-lingual retrieval approach transfers
before we run on the true unseen test languages.

In [ ]:
PROXY_LANG = 'tel'   # withheld from few-shot pool — acts as proxy for unseen langs

proxy_df   = train_df[train_df['language'] == PROXY_LANG].copy()
pool_df    = train_df[train_df['language'] != PROXY_LANG].copy()

# Re-embed pool (excluding proxy language)
print(f'Pool size (excl. {PROXY_LANG}): {len(pool_df):,}  |  Proxy size: {len(proxy_df):,}')
pool_passages   = [make_passage(r) for _, r in pool_df.iterrows()]
pool_embeddings = embedder.encode(
    pool_passages, batch_size=64, show_progress_bar=True,
    normalize_embeddings=True, convert_to_numpy=True
)

def retrieve_from_pool(query_row, k=3):
    q_emb  = embedder.encode([make_query(query_row)], normalize_embeddings=True, convert_to_numpy=True)
    scores = pool_embeddings @ q_emb.T
    top_k  = scores.flatten().argsort()[-k:][::-1]
    return [pool_df.iloc[i].to_dict() for i in top_k]

print('Proxy validation pool ready.')

In [ ]:
# Sample 50 proxy questions for speed
proxy_sample = proxy_df.sample(min(50, len(proxy_df)), random_state=SEED).to_dict('records')
proxy_preds  = []

if BACKEND == 'local' and model is not None:
    for i in range(0, len(proxy_sample), BATCH_SIZE):
        batch   = proxy_sample[i:i+BATCH_SIZE]
        prompts = [build_prompt(r, retrieve_from_pool(r, FEW_SHOT_K)) for r in batch]
        raws    = predict_local_batch(prompts)
        proxy_preds.extend(extract_answer(r) for r in raws)
elif BACKEND == 'openai':
    for r in proxy_sample:
        proxy_preds.append(extract_answer(predict_openai(build_prompt(r, retrieve_from_pool(r, FEW_SHOT_K)))))
elif BACKEND == 'anthropic':
    for r in proxy_sample:
        proxy_preds.append(extract_answer(predict_anthropic(build_prompt(r, retrieve_from_pool(r, FEW_SHOT_K)))))
else:
    print('Demo mode — random proxy predictions.')
    proxy_preds = list(np.random.choice(['A','B','C','D'], len(proxy_sample)))

proxy_acc = np.mean([p == r['answer'] for p, r in zip(proxy_preds, proxy_sample)])
print(f'Proxy ({PROXY_LANG}) accuracy: {proxy_acc:.4f} ({proxy_acc*100:.1f}%)')
print(f'Baseline (all-A):          ~0.25')
assert proxy_acc > 0.25, 'Cross-lingual approach is no better than random — check model/retrieval'

## 6. Predict on Unseen Test Languages

In [ ]:
test_rows   = test_df.to_dict('records')
final_preds = []

print(f'Predicting {len(test_rows):,} unseen-language questions...')
print(f'Languages: {sorted(test_df["language"].unique())}\n')

if BACKEND == 'local' and model is not None:
    for i in range(0, len(test_rows), BATCH_SIZE):
        batch   = test_rows[i:i+BATCH_SIZE]
        prompts = [build_prompt(r, retrieve_cross_lingual(r, FEW_SHOT_K)) for r in batch]
        raws    = predict_local_batch(prompts)
        for r, raw in zip(batch, raws):
            final_preds.append({'question_id': r['question_id'], 'answer': extract_answer(raw)})
        if (i // BATCH_SIZE) % 20 == 0:
            print(f'  {i+len(batch):,}/{len(test_rows):,}')
elif BACKEND == 'openai':
    for i, r in enumerate(test_rows):
        raw = predict_openai(build_prompt(r, retrieve_cross_lingual(r, FEW_SHOT_K)))
        final_preds.append({'question_id': r['question_id'], 'answer': extract_answer(raw)})
        if i % 100 == 0: print(f'  {i}/{len(test_rows)}...')
elif BACKEND == 'anthropic':
    for i, r in enumerate(test_rows):
        raw = predict_anthropic(build_prompt(r, retrieve_cross_lingual(r, FEW_SHOT_K)))
        final_preds.append({'question_id': r['question_id'], 'answer': extract_answer(raw)})
        if i % 100 == 0: print(f'  {i}/{len(test_rows)}...')
else:
    print('Demo mode — random predictions.')
    rng = random.Random(SEED)
    for r in test_rows:
        final_preds.append({'question_id': r['question_id'], 'answer': rng.choice(['A','B','C','D'])})

print(f'Done: {len(final_preds):,} predictions generated.')

## 7. Write Submission

In [ ]:
submission = pd.DataFrame(final_preds, columns=['question_id', 'answer'])

assert list(submission.columns) == ['question_id', 'answer']
assert submission['answer'].isin(['A','B','C','D']).all()
assert submission['question_id'].nunique() == len(submission)
assert len(submission) == len(test_df)

submission.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
print(f'Saved: {OUTPUT_PATH}  ({len(submission):,} rows)')
print('Answer distribution:')
print(submission['answer'].value_counts().to_string())

## Summary

| Step | Approach |
|---|---|
| Embedding | `multilingual-e5-small` maps all Indic scripts to shared space |
| Retrieval | Cosine similarity finds seen-language analogues for each test question |
| Prompting | 3 cross-lingual few-shot examples + domain header |
| Validation | Telugu held out as proxy unseen language; must beat 0.25 |
| Submission | `question_id`, `answer` — validated before write |

**Expected score**: ~0.45–0.55 with a 3B multilingual model.

**To improve further**:
- Translate test questions to Hindi (closest high-resource seen language) before prompting (+8–12%)
- Use `multilingual-e5-large` embeddings for higher-quality retrieval (+3–5%)
- Ensemble: majority vote across 5 temperature=0.7 passes (+2–4%)
- Fine-tune on seen languages with LoRA, then zero-shot on unseen (+5–10%)